In [16]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

download count data and meta data from [Open Targets](https://www.opentargets.org/projects/effectorness).

In [17]:
ad = sc.read_10x_mtx(
    'NCOMMS-19-7936188',
    var_names='gene_symbols'
)
meta = pd.read_csv('NCOMMS-19-7936188_metadata.txt', sep='\t')
ad.obs = ad.obs.join(meta)
# ad.write_h5ad('preprocessed.h5ad', compression='gzip')

In [6]:
meta

,cell.type,cytokine.condition,donor.id,batch.10X,nGene,nUMI,percent.mito,S.Score,G2M.Score,Phase,cluster.id,effectorness
N_resting_AAACCTGAGCTGTCTA,Naive,UNS,D4,2,1163,4172,0.023496,-0.134199,-0.159211,G1,TN (resting),0.151812
N_resting_AAACCTGTCACCACCT,Naive,UNS,D4,2,1037,3690,0.020867,-0.101756,-0.203707,G1,TN (resting),0.031763
N_resting_AAACCTGTCCGTTGTC,Naive,UNS,D2,2,1245,4446,0.027903,-0.145131,-0.164210,G1,TN (resting),0.113897
N_resting_AAACGGGAGGGTTCCC,Naive,UNS,D4,2,1016,3913,0.011509,-0.069492,-0.190810,G1,TN (resting),0.341240
N_resting_AAACGGGCAACAACCT,Naive,UNS,D1,2,1005,3557,0.039640,-0.124007,-0.143379,G1,TN (resting),0.019741
...,...,...,...,...,...,...,...,...,...,...,...,...
M_iTreg_r2_TTTGCGCGTGATAAGT,Memory,iTreg,D2,2,3491,14063,0.047003,-0.171570,-0.250832,G1,TEM (Th17/iTreg),0.894785
M_iTreg_r2_TTTGGTTCATGATCCA,Memory,iTreg,D3,2,4525,17509,0.040734,-0.090219,-0.263634,G1,TEM (Th17/iTreg),0.443469
M_iTreg_r2_TTTGGTTTCCTGTACC,Memory,iTreg,D1,2,3381,16735,0.022172,0.121549,-0.133797,S,TEM (Th17/iTreg),0.380790
M_iTreg_r2_TTTGGTTTCGGCTTGG,Memory,iTreg,D3,2,3690,16720,0.036787,-0.180000,-0.279017,G1,TEM (Th17/iTreg),0.758033


In [15]:
meta.groupby(['cell.type', 'cluster.id']).size().unstack(fill_value=0).T

cell.type,Memory,Naive
cluster.id,,
HSP.high,200,974
IFN.high,626,692
Mitotic,946,704
TCM (resting),1743,172
TCM1 (Th0),1176,733
TCM1 (Th17/iTreg),941,100
TCM2 (Th0),2529,67
TCM2 (Th17/iTreg),3795,65
TEM (Th0),1297,0


In [14]:
meta[meta['cell.type']=='Naive'].groupby(['cytokine.condition', 'cluster.id']).size().unstack(fill_value=0).T

cytokine.condition,Th0,Th17,Th2,UNS,iTreg
cluster.id,,,,,
HSP.high,125,154,160,0,535
IFN.high,102,279,126,0,185
Mitotic,51,184,312,0,157
TCM (resting),0,0,0,172,0
TCM1 (Th0),258,19,423,0,33
TCM1 (Th17/iTreg),20,64,2,0,14
TCM2 (Th0),30,7,21,0,9
TCM2 (Th17/iTreg),2,46,1,0,16
TEM (Th17/iTreg),2,71,2,0,37


In [6]:
# adata = sc.read_h5ad('preprocessed.h5ad')
adata = ad[ad.obs['cytokine.condition'] != 'UNS'].copy()

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=50)
adata.layers['raw'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=5000, flavor='seurat', subset=False)
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)
sc.tl.umap(adata)

In [ ]:
adata.obs['perturbation'] = np.where(adata.obs['cytokine.condition'] == 'Th0', 'control', 'unknown')

In [ ]:
adata.write_h5ad('preprocessed.h5ad', compression='gzip')

## preprare source cells and target cells

In [4]:
adata = sc.read_h5ad('preprocessed.h5ad', backed='r')
split_df = adata.obs[['perturbation', 'cell.type']].copy().rename(columns={'cell.type': 'subsplit'})
split_df['split'] = 'pred'
split_df.to_parquet('split_df_pred.parquet')

In [ ]:
## load the training split_df from schmidt dataset
split_df_train = pd.read_csv('../schmidt/split_trainonstimulated.csv').query('subsplit == "Re-stimulated"').set_index('cell')
split_df_train.replace({'split': {'test': 'val', 'val': 'train'}}, inplace=True)
split_df_train.to_parquet('split_df_train.parquet')

In [ ]:
adata = sc.read_h5ad('preprocessed.h5ad', backed='r')
split_df = adata.obs[['perturbation', 'cell.type']].copy().rename(columns={'cell.type': 'subsplit'})
split_df['split'] = 'pred'
split_df.to_parquet('split_df_pred.parquet')